# Análise Topológica e Centralidades da Malha Aérea Brasileira

## Introdução
Este notebook explora a estrutura da malha aérea comercial brasileira antes do fechamento do Aeroporto Salgado Filho (POA), utilizando dados de Abril de 2024. 

O objetivo principal aqui é investigar as **propriedades topológicas da rede** usando as ferramentas da disciplina de Algoritmos e Estruturas de Dados II. Vamos responder a três pontos centrais do nosso escopo:
1. **Hubs vs. Pontes (Hipótese 2):** Os aeroportos com mais voos são necessariamente os mais críticos para manter o país conectado?
2. **Estrutura Núcleo-Periferia (Subpergunta 1):** Existe um "núcleo duro" de aeroportos que sustentam a malha?
3. **Acessibilidade Regional (Subpergunta 5):** Qual a distância (em escalas) de Natal (SBSG) para o resto do país?

Esses dados servirão como nossa **linha de base** (baseline) antes de iniciarmos as simulações de ataques e fechamentos climáticos.

In [6]:
import sys
from pathlib import Path

# Configuração para importar os módulos da pasta src/
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(raiz / "src"))

import pandas as pd
from IPython.display import display

# Importando nossas funções personalizadas
from rede import carregar_grafo, nome_curto
from metricas import calcular_centralidades, tabela_graus, analise_kcore, escalas_bfs_origem, eficiencia_global_dirigida

# Carregar a malha de referência (Abril/2024)
caminho_grafo = raiz / "data" / "processed" / "rede_referencia.graphml"
G_ref = carregar_grafo(caminho_grafo)

print(f"Malha de Referência carregada: {G_ref.number_of_nodes()} aeroportos e {G_ref.number_of_edges()} rotas.")

Malha de Referência carregada: 154 aeroportos e 816 rotas.


## 1. Hubs vs. Pontes: Testando a Hipótese 2

Na teoria de redes, um aeroporto com muitas conexões diretas é chamado de **Hub** (alto Grau). No entanto, um aeroporto pode ter poucas rotas, mas ser a única ligação entre uma região inteira e o resto do país, funcionando como uma **Ponte** (alta *Betweenness Centrality*).

**Nossa Hipótese 2 (H2):** Existem aeroportos com *betweenness* alta e grau baixo/médio, que funcionam como pontes regionais e não aparecem nos rankings de maior movimentação.

In [2]:
# Calcular centralidades e graus
df_graus = tabela_graus(G_ref)
df_cents = calcular_centralidades(G_ref)

# Unir as tabelas para comparação
df_topologia = df_graus.join(df_cents)
df_topologia.insert(0, "aeroporto", [nome_curto(G_ref, n) for n in df_topologia.index])

# Exibir os 10 maiores Hubs (Grau Total)
print("✈️ TOP 10 AEROPORTOS POR GRAU TOTAL (HUBS):")
hubs = df_topologia.sort_values(by="grau total", ascending=False)
display(hubs[["aeroporto", "grau total", "betweenness"]].head(10))

# Exibir as 10 maiores Pontes (Betweenness Centrality)
print("\n🌉 TOP 10 AEROPORTOS POR BETWEENNESS (PONTES CRÍTICAS):")
pontes = df_topologia.sort_values(by="betweenness", ascending=False)
display(pontes[["aeroporto", "betweenness", "grau total"]].head(10))

✈️ TOP 10 AEROPORTOS POR GRAU TOTAL (HUBS):


,aeroporto,grau total,betweenness
SBKP,Viracopos (SP),112,0.236690
SBGR,Guarulhos (SP),104,0.152113
SBCF,Tancredo Neves (MG),102,0.227976
SBSP,Congonhas (SP),80,0.072870
SBRF,Guararapes (PE),72,0.151196
SBBR,Brasília (DF),72,0.098525
SBGL,Galeão (RJ),46,0.036033
SBBE,Val de Cans (PA),45,0.122684
SBEG,Eduardo Gomes (AM),45,0.211188
SBPA,Salgado Filho (RS),40,0.086265



🌉 TOP 10 AEROPORTOS POR BETWEENNESS (PONTES CRÍTICAS):


,aeroporto,betweenness,grau total
SBKP,Viracopos (SP),0.236690,112
SBCF,Tancredo Neves (MG),0.227976,102
SBEG,Eduardo Gomes (AM),0.211188,45
SBGR,Guarulhos (SP),0.152113,104
SBRF,Guararapes (PE),0.151196,72
SBBE,Val de Cans (PA),0.122684,45
SBCY,Marechal Rondon (MT),0.098953,34
SBBR,Brasília (DF),0.098525,72
SBPA,Salgado Filho (RS),0.086265,40
SBSP,Congonhas (SP),0.072870,80


## 2. Estrutura Núcleo-Periferia (Subpergunta 1)

Usando a decomposição **k-core**, podemos mapear a robustez estrutural da malha. O k-core vai removendo iterativamente os aeroportos com menos conexões até sobrar apenas o "núcleo duro" mais denso e coeso da aviação nacional.

In [7]:
# Aplicar a análise de k-core
df_core = analise_kcore(G_ref)
df_core.insert(0, "aeroporto", [nome_curto(G_ref, n) for n in df_core.index])

max_core = df_core["core_number"].max()
nucleo_duro = df_core[df_core["core_number"] == max_core]

print(f"🎯 O k-core máximo da malha é {max_core}.")
print(f"Isso significa que existe um núcleo de {len(nucleo_duro)} aeroportos onde todos estão conectados a pelo menos outros {max_core} aeroportos do próprio núcleo.")
display(nucleo_duro)

🎯 O k-core máximo da malha é 9.
Isso significa que existe um núcleo de 16 aeroportos onde todos estão conectados a pelo menos outros 9 aeroportos do próprio núcleo.


,aeroporto,core_number
SBFZ,Pinto Martins (CE),9
SBKP,Viracopos (SP),9
SBGR,Guarulhos (SP),9
SBBR,Brasília (DF),9
SBGL,Galeão (RJ),9
SBCF,Tancredo Neves (MG),9
SBRF,Guararapes (PE),9
SBSG,Governador Aluizio Alves (RN),9
SBSV,Deputado Luís Eduardo Magalhães (BA),9
SBSP,Congonhas (SP),9


## 3. Acessibilidade Regional: O caso de Natal (Subpergunta 5)

Para entender o impacto de rotas na experiência do passageiro, calculamos o número mínimo de escalas necessárias para viajar de um ponto a outro. Usamos o algoritmo de **Busca em Largura (BFS)** a partir do aeroporto de Natal (SBSG).

In [8]:
# Calcular distâncias a partir de Natal (SBSG) usando BFS
df_natal = escalas_bfs_origem(G_ref, "SBSG")
df_natal.insert(1, "nome_destino", df_natal["destino"].apply(lambda x: nome_curto(G_ref, x)))

# Filtrar algumas capitais estratégicas para visualização
capitais_interesse = ["SBBR", "SBGR", "SBGL", "SBCF", "SBEG", "SBFZ", "SBPA"]
escalas_capitais = df_natal[df_natal["destino"].isin(capitais_interesse)].copy()

print("🗺️ ESCALAS NECESSÁRIAS PARTINDO DE NATAL (SBSG):")
display(escalas_capitais.sort_values(by="escalas"))

# Resumo geral da acessibilidade de Natal
voos_diretos = len(df_natal[df_natal["escalas"] == 0])
uma_escala = len(df_natal[df_natal["escalas"] == 1])
duas_escalas = len(df_natal[df_natal["escalas"] == 2])

print(f"\nResumo a partir de SBSG:")
print(f"- Capitais/Cidades a 1 voo direto (0 escalas): {voos_diretos}")
print(f"- Cidades a 1 escala: {uma_escala}")
print(f"- Cidades a 2 escalas: {duas_escalas}")

🗺️ ESCALAS NECESSÁRIAS PARTINDO DE NATAL (SBSG):


,destino,nome_destino,voos,escalas
1,SBBR,Brasília (DF),1,0
2,SBCF,Tancredo Neves (MG),1,0
5,SBFZ,Pinto Martins (CE),1,0
6,SBGL,Galeão (RJ),1,0
7,SBGR,Guarulhos (SP),1,0
18,SBEG,Eduardo Gomes (AM),2,1
39,SBPA,Salgado Filho (RS),2,1



Resumo a partir de SBSG:
- Capitais/Cidades a 1 voo direto (0 escalas): 16
- Cidades a 1 escala: 94
- Cidades a 2 escalas: 38


## 4. Linha de Base: Eficiência Global

Antes de simularmos o fechamento de aeroportos (como o caso de Porto Alegre nas enchentes), precisamos gravar a **Eficiência Global** da rede intacta. A Eficiência Global é a média do inverso das distâncias mais curtas. Se a rede perder rotas ou fragmentar, esse número cairá drasticamente.

In [9]:
# Calcular a eficiência global da malha de abril
eficiencia_intacta = eficiencia_global_dirigida(G_ref)

print(f"🌍 Eficiência Global da malha intacta (Abril/2024): {eficiencia_intacta:.4f}")

🌍 Eficiência Global da malha intacta (Abril/2024): 0.3909
